# Object Selection Example

This notebook looks at how you can apply selections on NanoAOD objects with Awkward Array. We'll do this in a single event, as well as across multiple events simultaneously.

As in the event counting notebook, you will need to have run the appropriate `voms-proxy-init` command to be able to access the root file in this notebook.

We'll start by getting the events from one file, like in the previous example.

In [1]:
import awkward as ak
import numpy as np
from coffea.nanoevents import NanoEventsFactory
from coffea.analysis_tools import PackedSelection

redirector = "root://cmsxrootd.fnal.gov//"
file = redirector+"/store/mc/RunIII2024Summer24NanoAODv15/DYto2Mu-2Jets_Bin-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v6/100000/005c5bf4-d3d3-4fc1-9df3-99c83183c5f5.root"
events = NanoEventsFactory.from_root(
        file+":Events",
    ).events()
print("Events loaded")

Events loaded


## One Event

Let's grab the first event, and then just its AK4 jets. We can see the $p_t$ of the jets in the event as before.

Note: You will almost certainly not have to deal with this in a real analysis, but here we multiply the $p_t$ by 1 to force the array to materialize. This is related to the way coffea avoids actually reading data until it needs to, which generally speeds up code.

In [2]:
event0 = events[0]
jets0 = event0.Jet
print(f"p_t of AK4 jets in first event: {jets0.pt*1}")

p_t of AK4 jets in first event: [30.7, 24.3, 22.9, 22.9, 21.8, 21.2, 15.9, 15.8]


Suppose we want only jets with at least 30 GeV of $p_t$. We can create a filter as follows. In our filter, an entry is True if the corresponding jet has > 30 GeV, and is False otherwise.

In [3]:
pt_filter0 = jets0.pt > 30
print(f"Our p_t filter for the first event: {pt_filter0}")

Our p_t filter for the first event: [True, False, False, False, False, False, False, False]


Now, we can filter out the jets we want. We'll print the filtered jets' $p_t$ to confirm that this worked.

In [4]:
filtered_jets0 = jets0[pt_filter0]
print(f"p_t of AK4 jets in first event after filtering: {filtered_jets0.pt}")
print(f"eta of AK4 jets in first event after filtering: {filtered_jets0.eta}")

p_t of AK4 jets in first event after filtering: [30.7]
eta of AK4 jets in first event after filtering: [??]


We can further on eta. Suppose we only want jets with $|\eta|$ < 2.0.

In [5]:
twice_filtered_jets0 = filtered_jets0[abs(filtered_jets0.eta) < 2.0]
print(f"p_t of AK4 jets in first event after filtering twice: {twice_filtered_jets0.pt}")
print(f"eta of AK4 jets in first event after filtering twice: {twice_filtered_jets0.eta}")

p_t of AK4 jets in first event after filtering twice: [30.7]
eta of AK4 jets in first event after filtering twice: [-1.26]


We also could have done those both at once by performing logical operations on the filters. For example, this gives the same set of jets as the above:

In [6]:
also_twice_filtered_jets0 = jets0[(jets0.pt > 30) & (abs(jets0.eta) < 2.0)]
print(f"p_t of AK4 jets in first event with both filters: {also_twice_filtered_jets0.pt}")
print(f"eta of AK4 jets in first event with both filters: {also_twice_filtered_jets0.eta}")

p_t of AK4 jets in first event with both filters: [30.7]
eta of AK4 jets in first event with both filters: [-1.26]


## Multiple Events

To apply the same selections on multiple events, we can use the same syntax as above, just replacing `event0` with `events`.

In [7]:
jets = events.Jet
pt_filter = jets.pt > 30
eta_filter = abs(jets.eta) < 2.0
filtered_jets = jets[pt_filter & eta_filter]

To see that this worked, let's look at the pt and eta of jets in the first event before and after filtering. It should be the same as what we got in the last section.

In [8]:
print(f"p_t of AK4 jets in first event before filtering: {jets[0].pt}")
print(f"eta of AK4 jets in first event before filtering: {jets[0].eta}")
print(f"p_t of AK4 jets in first event after filtering: {filtered_jets[0].pt}")
print(f"eta of AK4 jets in first event after filtering: {filtered_jets[0].eta}")

p_t of AK4 jets in first event before filtering: [30.7, 24.3, 22.9, 22.9, 21.8, 21.2, 15.9, 15.8]
eta of AK4 jets in first event before filtering: [-1.26, 2.73, 0.878, -0.423, -2.84, -0.862, -2.77, -2.97]
p_t of AK4 jets in first event after filtering: [30.7]
eta of AK4 jets in first event after filtering: [-1.26]


We also made the same selections in the second, and all other events. Looking at the second event:

In [9]:
print(f"p_t of AK4 jets in second event before filtering: {jets[1].pt}")
print(f"eta of AK4 jets in second event before filtering: {jets[1].eta}")
print(f"p_t of AK4 jets in second event after filtering: {filtered_jets[1].pt}")
print(f"eta of AK4 jets in second event after filtering: {filtered_jets[1].eta}")

p_t of AK4 jets in second event before filtering: [74.5, 64.1, 62.6, 57.5, 30, 16.2]
eta of AK4 jets in second event before filtering: [-0.0924, -1.74, 2.59, 1.18, 2.3, -1.63]
p_t of AK4 jets in second event after filtering: [74.5, 64.1, 57.5]
eta of AK4 jets in second event after filtering: [-0.0924, -1.74, 1.18]


## Masking

Awkward array provides another way to selection objects, but without losing information about where the dropped object were, or how many were dropped. This is with `ak.mask`.

Instead of dropping jets that don't meet the requirements, they are replaced with None. This can become useful when combining multiple selections for more complicated requirements. We give an example of this with the first event in our file.

In [10]:
jets0 = events.Jet[0]
pt_filter0 = jets0.pt > 30
masked_jets0 = ak.mask(jets0,pt_filter0)
print(f"Masked array of jets in event 1: {masked_jets0}")
print(f"pt of the masked array of jets in event 1: {masked_jets0.pt}")

Masked array of jets in event 1: [Jet, ...]
pt of the masked array of jets in event 1: [30.7, None, None, None, None, None, None, None]
